# Social signals and algorithmic trading of Bitcoin

**Authors:** David García, Frank Schweitzer

**Published:** 2015-09-01

**URL:** [https://doi.org/10.1098/rsos.150288](https://doi.org/10.1098/rsos.150288)

**Abstract:**
The availability of data on digital traces is growing to unprecedented sizes, but inferring actionable knowledge from large-scale data is far from being trivial. This is especially important for computational finance, where digital traces of human behaviour offer a great potential to drive trading strategies. We contribute to this by providing a consistent approach that integrates various datasources in the design of algorithmic traders. This allows us to derive insights into the principles behind the profitability of our trading strategies. We illustrate our approach through the analysis of Bitcoin, a cryptocurrency known for its large price fluctuations. In our analysis, we include economic signals of volume and price of exchange for USD, adoption of the Bitcoin technology and transaction volume of Bitcoin. We add social signals related to information search, word of mouth volume, emotional valence and opinion polarization as expressed in tweets related to Bitcoin for more than 3 years. Our analysis reveals that increases in opinion polarization and exchange volume precede rising Bitcoin prices, and that emotional valence precedes opinion polarization and rising exchange volumes. We apply these insights to design algorithmic trading strategies for Bitcoin, reaching very high profits in less than a year. We verify this high profitability with robust statistical methods that take into account risk and trading costs, confirming the long-standing hypothesis that trading-based social media sentiment has the potential to yield positive returns on investment.

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

In [ ]:
# Phase 1: Configuration
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# Set display options
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-darkgrid')

In [ ]:
# Phase 2: Data Download and Feature Engineering
data = yf.download('BTC-USD', start='2020-01-01', end='2023-01-01')
data['Returns'] = data['Adj Close'].pct_change()
data['Volatility'] = data['Returns'].rolling(window=21).std() * np.sqrt(252)
data.dropna(inplace=True)
data.head()

In [ ]:
# Phase 3: Signal Generation and Portfolio Construction
# Simple moving average crossover strategy
short_window = 40
long_window = 100

data['Short_MA'] = data['Adj Close'].rolling(window=short_window, min_periods=1).mean()
data['Long_MA'] = data['Adj Close'].rolling(window=long_window, min_periods=1).mean()
data['Signal'] = 0

data.loc[data['Short_MA'] > data['Long_MA'], 'Signal'] = 1
data.loc[data['Short_MA'] < data['Long_MA'], 'Signal'] = -1
data['Position'] = data['Signal'].shift(1)
data.dropna(inplace=True)
data.head()

In [ ]:
# Phase 4: Vectorized Backtest
# Calculate strategy returns
data['Strategy_Returns'] = data['Position'] * data['Returns']
data['Cumulative_Strategy_Returns'] = (1 + data['Strategy_Returns']).cumprod()
data['Cumulative_Market_Returns'] = (1 + data['Returns']).cumprod()
data.tail()

In [ ]:
# Phase 5: Performance Metrics
# Sharpe Ratio
sharpe_ratio = data['Strategy_Returns'].mean() / data['Strategy_Returns'].std() * np.sqrt(252)

# Sortino Ratio
sortino_ratio = data['Strategy_Returns'].mean() / data[data['Strategy_Returns'] < 0]['Strategy_Returns'].std() * np.sqrt(252)

# Calmar Ratio
max_drawdown = (data['Cumulative_Strategy_Returns'].cummax() - data['Cumulative_Strategy_Returns']).max()
calmar_ratio = data['Strategy_Returns'].mean() / max_drawdown

# Plot equity curve
data[['Cumulative_Strategy_Returns', 'Cumulative_Market_Returns']].plot(figsize=(14, 7))
plt.title('Equity Curve')
plt.show()

# Print metrics
print(f'Sharpe Ratio: {sharpe_ratio:.2f}')
print(f'Sortino Ratio: {sortino_ratio:.2f}')
print(f'Calmar Ratio: {calmar_ratio:.2f}')
print(f'Max Drawdown: {max_drawdown:.2%}')

In [ ]:
# Phase 6: Monitoring Stub
# Placeholder for monitoring logic
# This could involve setting up alerts, logging, or real-time data feeds
print('Monitoring phase is a stub and requires implementation.')